# XDP FNV N-Gram Classifier Training

This notebook mirrors the methodology from `xdp_ngram_classifier_with_vsr_data.ipynb`, but replaces sklearn `HashingVectorizer` with the exact FNV-style byte hash used by `xdp_ngram_hash3()` in the XDP program.

It also adds a router-intent training layer. The public datasets provide useful background, but the kernel router needs examples that look like real prompts: software tasks should route to `coding`, deduction/constraint tasks to `reasoning`, and explanatory/everyday prompts to `general`.

The final export is intended for the `xdp_ngram_weights` BPF map.

In [ ]:
# Install once if needed:
# %pip install datasets pandas numpy scipy scikit-learn

import json
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset
from scipy.sparse import csr_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
FINAL_NGRAM_RANGE = (3, 3)
FINAL_FEATURES = 4096
MAX_ABS_WEIGHT = 127
ROUTER_INTENT_WEIGHT = 6.0
OUTPUT_PATH = Path("xdp_ngram_model_fnv.json")

## 1. Load and Prepare a Routing Dataset

This uses the same VSR supplement + MMLU-Pro dataset mix and category-to-route mapping as the original VSR notebook.

In [ ]:
CATEGORY_TO_ROUTE = {
    "computer science": "coding",

    "math": "reasoning",
    "physics": "reasoning",
    "engineering": "reasoning",
    "chemistry": "reasoning",

    "biology": "general",
    "business": "general",
    "economics": "general",
    "health": "general",
    "history": "general",
    "law": "general",
    "other": "general",
    "philosophy": "general",
    "psychology": "general",
}


def dataset_dict_to_frame(dataset_dict, text_column, label_column):
    frames = []

    for split_name, split in dataset_dict.items():
        frame = split.to_pandas()[[text_column, label_column]].copy()
        frame.columns = ["text", "category"]
        frame["source_split"] = split_name
        frames.append(frame)

    return pd.concat(frames, ignore_index=True)


# Conversational and non-academic routing examples.
supplement_ds = load_dataset("llm-semantic-router/category-classifier-supplement")
supplement_df = dataset_dict_to_frame(
    supplement_ds,
    text_column="text",
    label_column="label",
)

# Harder academic questions across the same 14 categories.
mmlu_ds = load_dataset("TIGER-Lab/MMLU-Pro")
mmlu_df = dataset_dict_to_frame(
    mmlu_ds,
    text_column="question",
    label_column="category",
)

df = pd.concat([supplement_df, mmlu_df], ignore_index=True)

df["category"] = df["category"].astype(str).str.strip().str.lower()
df["text"] = df["text"].astype(str).str.strip()
df["label"] = df["category"].map(CATEGORY_TO_ROUTE)

df = (
    df.dropna(subset=["text", "label"])
      .drop_duplicates(subset=["text"])
      .reset_index(drop=True)
)

df["sample_weight"] = 1.0

print("Rows:", len(df))
print("Route distribution:")
print(df["label"].value_counts())
print("Original category distribution:")
print(df["category"].value_counts())

df.sample(10, random_state=RANDOM_STATE)[["text", "category", "label"]]

## 2. Add Router-Intent Examples

The public datasets are topic-oriented. This augmentation makes the labels match router behavior: `coding` means developer work, `reasoning` means explicit multi-step reasoning, and `general` means explanation/summarization/everyday knowledge even when the topic is STEM-adjacent.

In [ ]:
def expand_templates(templates, slots, label):
    rows = []
    for template in templates:
        keys = [part[1] for part in __import__("string").Formatter().parse(template) if part[1]]
        if not keys:
            rows.append({"text": template, "category": "router_intent", "label": label, "source_split": "curated_router"})
            continue

        def build(index, values):
            if index == len(keys):
                rows.append({
                    "text": template.format(**values),
                    "category": "router_intent",
                    "label": label,
                    "source_split": "curated_router",
                })
                return
            key = keys[index]
            for value in slots[key]:
                values[key] = value
                build(index + 1, values)

        build(0, {})
    return rows


coding_templates = [
    "Debug this {language} error: {error}.",
    "Write a {language} function that {task}.",
    "Refactor this {language} code to improve {quality}.",
    "Create tests for a {language} module that {task}.",
    "Explain why my {language} {component} is {failure_mode}.",
    "Optimize a {language} implementation of {algorithm}.",
]

coding_slots = {
    "language": ["Python", "JavaScript", "TypeScript", "Rust", "Go", "C"],
    "error": [
        "TypeError in my code", "undefined is not a function", "segmentation fault",
        "borrowed value does not live long enough", "index out of range",
        "deadlock while reading from a channel",
    ],
    "task": [
        "parses JSON safely", "deduplicates records", "validates an email address",
        "streams a file line by line", "retries failed HTTP requests", "computes rolling averages",
    ],
    "quality": ["readability", "performance", "error handling", "testability"],
    "component": ["API handler", "parser", "CLI tool", "worker", "cache layer"],
    "failure_mode": ["timing out", "returning null", "dropping messages", "using too much memory"],
    "algorithm": ["binary search", "quicksort", "Dijkstra's algorithm", "LRU caching"],
}

reasoning_templates = [
    "Solve this logic puzzle step by step: {puzzle}.",
    "Given these constraints, determine the best option: {constraints}.",
    "Prove or disprove this claim: {claim}.",
    "Work through this probability question carefully: {probability}.",
    "Analyze the tradeoffs and choose a strategy for {scenario}.",
    "Find the hidden assumption in this argument: {argument}.",
]

reasoning_slots = {
    "puzzle": [
        "three people each make one true statement and one false statement",
        "five boxes are ordered by weight using only pairwise comparisons",
        "a schedule must fit four meetings into three rooms",
        "two guards give opposite answers about which door is safe",
    ],
    "constraints": [
        "minimize cost while keeping latency under 100 ms",
        "maximize reliability with only two backup systems",
        "choose a route with time, toll, and fuel constraints",
        "assign tasks when each person has a different unavailable day",
    ],
    "claim": [
        "a greedy algorithm always finds the optimal schedule",
        "correlation is enough to establish causation in this dataset",
        "if all premises are true, the conclusion must be true",
        "this theorem follows by induction",
    ],
    "probability": [
        "drawing two matching socks from a drawer with mixed colors",
        "rolling at least one six with three dice",
        "choosing a defective item after two inspection stages",
        "updating a belief after a positive medical test",
    ],
    "scenario": [
        "allocating compute across three model endpoints",
        "prioritizing bug fixes before a release",
        "choosing between speed, accuracy, and cost",
        "planning a migration with rollback risk",
    ],
    "argument": [
        "this product is popular, so it must be the best",
        "the experiment succeeded once, so the system is reliable",
        "the cheaper option saves money, so it is always preferable",
        "the model answered confidently, so the answer is correct",
    ],
}

general_templates = [
    "Explain {topic} in simple terms.",
    "Summarize the main benefits and drawbacks of {topic}.",
    "Give me a practical checklist for planning {activity}.",
    "Compare {topic} and {comparison} for everyday use.",
    "Write a concise overview of {topic} with examples.",
    "What should I know before starting {activity}?",
]

general_slots = {
    "topic": [
        "renewable energy", "remote work", "electric vehicles", "personal budgeting",
        "urban gardening", "sleep hygiene", "public transportation", "meal planning",
        "digital privacy", "climate change", "photosynthesis", "basic physics",
    ],
    "comparison": [
        "traditional commuting", "gas cars", "manual tracking", "buying groceries daily",
        "paper notes", "subscription services", "older energy sources", "a beginner's guide",
    ],
    "activity": [
        "a weekend trip", "a home office", "a study schedule", "a monthly budget",
        "a small garden", "a fitness routine", "a renewable energy project", "a science fair talk",
    ],
}

curated_rows = []
curated_rows.extend(expand_templates(coding_templates, coding_slots, "coding"))
curated_rows.extend(expand_templates(reasoning_templates, reasoning_slots, "reasoning"))
curated_rows.extend(expand_templates(general_templates, general_slots, "general"))
curated_df = pd.DataFrame(curated_rows).drop_duplicates(subset=["text"]).reset_index(drop=True)
curated_df["sample_weight"] = ROUTER_INTENT_WEIGHT

# Keep public data as background and upweight route-intent prompts directly.
base_df = df.copy()
df = pd.concat([base_df, curated_df], ignore_index=True)
df = df.drop_duplicates(subset=["text", "label", "source_split"]).reset_index(drop=True)

print("Base rows:", len(base_df))
print("Curated unique rows:", len(curated_df))
print("Training rows:", len(df))
print("Router-intent sample weight:", ROUTER_INTENT_WEIGHT)
print("\nTraining route distribution:")
print(df["label"].value_counts())
print("\nCurated route distribution:")
print(curated_df["label"].value_counts())
curated_df.sample(10, random_state=RANDOM_STATE)[["text", "label"]]

## 3. Build XDP FNV Character N-Gram Features

The final kernel classifier currently uses 3-byte character n-grams. The comparison cells below also support 4-grams and 3-5 grams to mirror the original notebook's model-selection workflow.

In [ ]:
FNV_OFFSET = 2166136261
FNV_PRIME = 16777619


def xdp_hash_bytes(raw, start, n, n_features):
    value = FNV_OFFSET
    for offset in range(n):
        value ^= raw[start + offset]
        value = (value * FNV_PRIME) & 0xFFFFFFFF
    return value & (n_features - 1)


def xdp_hash3(c0, c1, c2, n_features=FINAL_FEATURES):
    value = FNV_OFFSET
    for char in (c0, c1, c2):
        value ^= char
        value = (value * FNV_PRIME) & 0xFFFFFFFF
    return value & (n_features - 1)


def xdp_vectorize(texts, ngram_range=(3, 3), n_features=4096):
    min_n, max_n = ngram_range
    rows = []
    cols = []
    data = []

    for row, text in enumerate(texts):
        raw = str(text).lower().encode("utf-8", errors="ignore")
        counts = {}

        for n in range(min_n, max_n + 1):
            if len(raw) < n:
                continue
            for i in range(len(raw) - n + 1):
                feature = xdp_hash_bytes(raw, i, n, n_features)
                counts[feature] = counts.get(feature, 0) + 1

        for feature, count in counts.items():
            rows.append(row)
            cols.append(feature)
            data.append(count)

    return csr_matrix((data, (rows, cols)), shape=(len(texts), n_features), dtype=np.float32)


def train_model(texts, labels, sample_weight=None, ngram_range=(3, 3), n_features=4096):
    X = xdp_vectorize(texts, ngram_range=ngram_range, n_features=n_features)
    if sample_weight is None:
        sample_weight = np.ones(len(texts), dtype=np.float32)

    X_train, X_test, y_train, y_test, weight_train, weight_test = train_test_split(
        X,
        labels,
        sample_weight,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=labels,
    )

    model = LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
    )
    model.fit(X_train, y_train, sample_weight=weight_train)

    predictions = model.predict(X_test)

    return {
        "model": model,
        "accuracy": accuracy_score(y_test, predictions),
        "macro_f1": f1_score(y_test, predictions, average="macro"),
        "report": classification_report(y_test, predictions, zero_division=0),
        "confusion_matrix": confusion_matrix(
            y_test,
            predictions,
            labels=model.classes_,
        ),
        "classes": model.classes_,
    }


print("hash('con') =", xdp_hash3(ord('c'), ord('o'), ord('n')))
print("hash('bug') =", xdp_hash3(ord('b'), ord('u'), ord('g')))

## 4. Compare 3-Grams, 4-Grams, and 3-5 Grams

In [ ]:
results = []

for ngram_range in [(3, 3), (4, 4), (3, 5)]:
    result = train_model(
        df["text"],
        df["label"],
        sample_weight=df["sample_weight"],
        ngram_range=ngram_range,
        n_features=4096,
    )
    results.append({
        "ngram_range": str(ngram_range),
        "accuracy": result["accuracy"],
        "macro_f1": result["macro_f1"],
    })

pd.DataFrame(results).sort_values("macro_f1", ascending=False)

## 5. Test Fixed XDP-Style Prefix Lengths

In [ ]:
def byte_prefix(text, max_bytes):
    raw = text.encode("utf-8")[:max_bytes]
    return raw.decode("utf-8", errors="ignore")


prefix_results = []

for prefix_size in [64, 128, 200, 512, 1024, 2056]:
    texts = df["text"].map(lambda text: byte_prefix(text, prefix_size))

    result = train_model(
        texts,
        df["label"],
        sample_weight=df["sample_weight"],
        ngram_range=(3, 3),
        n_features=4096,
    )

    prefix_results.append({
        "prefix_bytes": prefix_size,
        "accuracy": result["accuracy"],
        "macro_f1": result["macro_f1"],
    })

pd.DataFrame(prefix_results)

## 6. Simulate TCP Segmentation

The current XDP implementation keeps rolling n-gram state in the flow map, so n-grams crossing packet boundaries are preserved for in-order packets. This cell simulates segmented scanning while preserving that rolling state.

In [ ]:
def split_bytes(text, segment_size):
    raw = text.encode("utf-8")
    return [raw[i:i + segment_size] for i in range(0, len(raw), segment_size)]


def xdp_vectorize_segmented(texts, segment_size, ngram_range=(3, 3), n_features=4096):
    min_n, max_n = ngram_range
    rows = []
    cols = []
    data = []

    for row, text in enumerate(texts):
        counts = {}
        tail = b""

        for segment in split_bytes(str(text).lower(), segment_size):
            raw = tail + segment
            base_skip = len(tail)

            for n in range(min_n, max_n + 1):
                if len(raw) < n:
                    continue
                for i in range(len(raw) - n + 1):
                    if i + n <= base_skip:
                        continue
                    feature = xdp_hash_bytes(raw, i, n, n_features)
                    counts[feature] = counts.get(feature, 0) + 1

            tail = raw[-(max_n - 1):] if max_n > 1 else b""

        for feature, count in counts.items():
            rows.append(row)
            cols.append(feature)
            data.append(count)

    return csr_matrix((data, (rows, cols)), shape=(len(texts), n_features), dtype=np.float32)


indices = np.arange(len(df))
X_full = xdp_vectorize(df["text"], ngram_range=(3, 3), n_features=4096)
X_train, X_test, y_train, y_test, weight_train, weight_test, train_idx, test_idx = train_test_split(
    X_full,
    df["label"],
    df["sample_weight"],
    indices,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df["label"],
)

model = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
)
model.fit(X_train, y_train, sample_weight=weight_train)

segmentation_results = []

test_texts = df.iloc[test_idx]["text"].tolist()

for segment_size in [32, 64, 128, 200, 512, 1024, 2056]:
    X_segmented = xdp_vectorize_segmented(test_texts, segment_size, ngram_range=(3, 3), n_features=4096)
    predictions = model.predict(X_segmented)

    segmentation_results.append({
        "segment_bytes": segment_size,
        "accuracy": accuracy_score(y_test, predictions),
        "macro_f1": f1_score(y_test, predictions, average="macro"),
    })

pd.DataFrame(segmentation_results)

## 7. Train a Final Model and Inspect Prompts

In [ ]:
X = xdp_vectorize(
    df["text"],
    ngram_range=FINAL_NGRAM_RANGE,
    n_features=FINAL_FEATURES,
)

final_model = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
)
final_model.fit(X, df["label"], sample_weight=df["sample_weight"])

smoke_cases = [
    ("Debug this Python TypeError in my code", "coding"),
    ("Write a Python function that parses JSON safely", "coding"),
    ("Refactor this Rust code to improve error handling", "coding"),
    ("Solve this logic puzzle step by step", "reasoning"),
    ("What is the capital of France?", "general"),
    ("Explain renewable energy in simple terms", "general"),
]

sample_prompts = [prompt for prompt, expected in smoke_cases]
expected_routes = [expected for prompt, expected in smoke_cases]
sample_X = xdp_vectorize(sample_prompts, ngram_range=FINAL_NGRAM_RANGE, n_features=FINAL_FEATURES)
sample_scores = final_model.decision_function(sample_X)
sample_predictions = final_model.predict(sample_X)

smoke_df = pd.DataFrame({
    "prompt": sample_prompts,
    "expected": expected_routes,
    "route": sample_predictions,
    "ok": sample_predictions == np.array(expected_routes),
    **{f"score_{name}": sample_scores[:, i] for i, name in enumerate(final_model.classes_)},
})

print("Smoke accuracy:", smoke_df["ok"].mean())
smoke_df

## 8. Quantize Weights for Integer-Only XDP Scoring

In [ ]:
def quantize_weights(model, max_abs_value=127):
    weights = model.coef_
    bias = model.intercept_

    largest = np.max(np.abs(weights))
    scale = max_abs_value / largest if largest > 0 else 1.0

    q_weights = np.round(weights * scale).astype(np.int16)
    q_bias = np.round(bias * scale).astype(np.int32)

    return q_weights, q_bias, scale


q_weights, q_bias, scale = quantize_weights(final_model, MAX_ABS_WEIGHT)

print("Classes:", final_model.classes_.tolist())
print("Weight shape:", q_weights.shape)
print("Bias shape:", q_bias.shape)
print("Scale:", scale)
print("Bias:", q_bias.tolist())

## 9. Export the FNV Model

In [ ]:
export = {
    "classes": final_model.classes_.tolist(),
    "ngram_range": list(FINAL_NGRAM_RANGE),
    "n_features": FINAL_FEATURES,
    "hash": "xdp_fnv_v1",
    "fnv_offset": FNV_OFFSET,
    "fnv_prime": FNV_PRIME,
    "scale": float(scale),
    "bias": q_bias.tolist(),
    "weights": q_weights.tolist(),
}

with OUTPUT_PATH.open("w") as file:
    json.dump(export, file)

print(f"Saved {OUTPUT_PATH}")

## Next Steps

1. Compare macro-F1 across n-gram ranges and prefix limits before exporting.
2. Inspect the smoke-test prompts; if coding/general/reasoning still miss your desired router behavior, add curated examples for software-engineering, logic-puzzle, and everyday-general prompts.
3. Copy or rename `xdp_ngram_model_fnv.json` to the path used by the XDP loader only after the smoke test looks reasonable.